# Phase 10.6 — Ranking Evaluation

Offline RRF-order vs fixed baseline ranking on `productiq_lexical_retrieval_v1` (no tuning).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.ranking.evaluation import RankingBenchmarkEvaluator, RankingEvaluationConfig
from productiq.retrieval.evaluation.dataset import default_benchmark_path, load_lexical_retrieval_benchmark

In [ ]:
benchmark = load_lexical_retrieval_benchmark(default_benchmark_path(ROOT))
config = RankingEvaluationConfig(evaluation_top_k=50, candidate_pool_top_k=50)
evaluator = RankingBenchmarkEvaluator(config)
{
    "benchmark_name": benchmark.benchmark_name,
    "benchmark_version": benchmark.benchmark_version,
    "query_count": len(benchmark.queries),
}

In [ ]:
# Artifact-style demo on one query (no live indexes required)
from productiq.ranking.evaluation import evaluate_ranking_query

q = benchmark.queries[0]
demo_pool = q.relevant_product_ids + ("decoy_1", "decoy_2")
row = evaluate_ranking_query(
    q,
    rrf_ordered_product_ids=demo_pool,
    baseline_ranked_product_ids=tuple(reversed(demo_pool)),
    filtered_pool_product_ids=demo_pool,
    config=config,
)
{
    "query_id": row.query_id,
    "candidate_coverage": row.candidate_coverage,
    "rrf_mrr": row.rrf.metrics.reciprocal_rank,
    "baseline_mrr": row.baseline.metrics.reciprocal_rank,
    "diagnostics": [tag.value for tag in row.diagnostics.diagnostic_tags],
}

Full benchmark runs require the production retrieval stack (indexes + catalog). Use `RankingBenchmarkEvaluator.evaluate_production_pipeline` with a wired `ProductionRetrievalPipeline` in that environment.